# H SC 790 · Lab 11 · Build the Harness Before You Trust the Model

**Week 12.** Due Thursday, April 22, 11:59 PM · 30 points.

**What earns full credit**
- Runs end to end from a fresh runtime
- Model, baseline, and second version all scored
- Injection and negation cases reported
- AI-use note included

**Rules.** Everything here is synthetic. Never load protected health information or confidential employer data into Colab or GitHub. The lab is finished when it runs top to bottom from a fresh runtime (Runtime → Restart session and run all).

The data dictionary for this lab is the README in the lab's folder of the course repository.

Cells marked `# YOUR CODE HERE` and *Your answer here.* are yours to complete. Add cells freely.

In [ ]:
# Setup: run this cell first, every time.
import os, json, urllib.request
import numpy as np
import pandas as pd

# The course data, read straight from the course repository. Never from your own machine.
COURSE_URL = os.environ.get("HSC790_DATA", "https://raw.githubusercontent.com/OWNER/hsc790-spr27/main/data/")

def read_text(rel):
    """Text of one file in the course data folder, for example read_text('SNAPSHOT.txt')."""
    return urllib.request.urlopen(COURSE_URL + rel).read().decode()

print("Data snapshot:", read_text("SNAPSHOT.txt").strip())

Three hundred free-text case reports, in English and Spanish. The task is to extract four fields (`hospitalized`, `onset_date`, `occupation`, `priority`) and to know, with numbers, how far to trust the result. You will build the harness first: gold labels and human agreement, a rule-based baseline, a validator, scoring, failure probes, and a regression test for a new model version.

**The model outputs in this lab are recorded and simulated.** `call_model` below replays what "version 1" or "version 2" returned for each report, so everyone scores the same outputs and the lab runs with no account or key. If your instructor has set up a live model, replace `call_model` with a real call and keep everything else; that is the point of a harness.

In [ ]:
import re
reports = pd.read_csv(COURSE_URL + "lab11/case_narratives.csv").set_index("report_id")
gold = pd.read_csv(COURSE_URL + "lab11/gold_labels.csv", dtype=str).set_index("report_id")
ann2 = pd.read_csv(COURSE_URL + "lab11/annotator2_labels.csv", dtype=str).set_index("report_id")
def read_jsonl(rel):
    return pd.DataFrame([json.loads(line) for line in read_text(rel).splitlines() if line]).set_index("report_id")
recorded = {"v1": read_jsonl("lab11/model_outputs_v1.jsonl"), "v2": read_jsonl("lab11/model_outputs_v2.jsonl")}
rerun = read_jsonl("lab11/model_outputs_v1_rerun.jsonl")
FIELDS = ["hospitalized", "onset_date", "occupation", "priority"]
print(read_text("lab11/annotation_guide.md")[:1500])

In [ ]:
MODEL_VERSION = "v1"
PROMPT = """Extract four fields from this case report and return one JSON object with keys hospitalized, onset_date, occupation,
priority, and evidence (a quote from the report for each field). Use "unknown" when the report does not say.

Report:
{report_text}"""

def call_model(prompt):
    """Recorded stand-in for a language model: returns what MODEL_VERSION answered for the report inside the prompt."""
    text = prompt.split("Report:\n", 1)[1]
    rid = reports.index[reports.text == text][0]
    return recorded[MODEL_VERSION].loc[rid, "raw_output"]

print(reports.text.iloc[0], "\n\n", call_model(PROMPT.format(report_text=reports.text.iloc[0])))

## 1. Read the gold set and measure agreement

Compute Cohen's kappa between the two annotators for each field, on the 100 reports both labeled. Which field is hardest for humans? Read some of the disagreements.

In [ ]:
def kappa(a, b):
    """Cohen's kappa for two aligned Series of labels."""
    # YOUR CODE HERE
    pass

# YOUR CODE HERE

## 2. Build the rule-based baseline

Regular expressions for each field, as in Lab 6. Score it against the gold labels. This is the bar a model has to beat.

In [ ]:
def prf(pred, truth, positive):
    tp = ((pred == positive) & (truth == positive)).sum()
    return pd.Series({"precision": tp / max(1, (pred == positive).sum()), "recall": tp / max(1, (truth == positive).sum())}).round(3)

def score(pred):
    """One row of scores for a DataFrame of predictions indexed by report_id (missing rows count as wrong)."""
    p = pred.reindex(gold.index)
    h, u = prf(p.hospitalized, gold.hospitalized, "yes"), prf(p.priority, gold.priority, "urgent")
    return pd.Series({"hosp_precision": h.precision, "hosp_recall": h.recall, "urgent_precision": u.precision, "urgent_recall": u.recall,
                      "occupation_accuracy": (p.occupation == gold.occupation).mean().round(3), "onset_exact": (p.onset_date == gold.onset_date).mean().round(3),
                      "onset_invented": int(((gold.onset_date == "unknown") & p.onset_date.notna() & (p.onset_date != "unknown")).sum())})

def baseline(text):
    """Rules only. Returns a dict with the four fields."""
    # YOUR CODE HERE
    pass

base_pred = pd.DataFrame({rid: baseline(t) for rid, t in reports.text.items()}).T
results = pd.DataFrame({"baseline": score(base_pred)})
results

## 3. Write the prompt and the validator

Structured output with "unknown" allowed and an evidence quote for each field. Parse, validate against the allowed values, and verify that every quote really is in the report. The slide's validator is below; extend `ALLOWED` to the other fields and check the date format.

In [ ]:
ALLOWED = {'hospitalized': {'yes', 'no', 'unknown'},
           'priority': {'urgent', 'routine'}}
# YOUR CODE HERE

def extract(report):
    raw = call_model(PROMPT.format(report_text=report))
    try:
        out = json.loads(raw)
    except ValueError:
        return None, 'not_json'
    for field, ok in ALLOWED.items():
        if out.get(field) not in ok:
            return None, f'bad_value:{field}'
    # YOUR CODE HERE
    pass
    for field, quote in out.get('evidence', {}).items():
        if quote and quote.lower() not in report.lower():
            return None, f'quote_not_found:{field}'
    return out, 'ok'

extract(reports.text.iloc[0])

## 4. Run and score

Run every report through `extract`. Report validation failures by kind, then precision and recall by field, overall and by language and report length. Decide what a failed validation counts as (here: wrong, because a person has to do that report by hand).

In [ ]:
def run(version):
    global MODEL_VERSION
    MODEL_VERSION = version
    res = {rid: extract(t) for rid, t in reports.text.items()}
    pred = pd.DataFrame({rid: (o or {}) for rid, (o, s) in res.items()}).T.reindex(reports.index)
    pred["status"] = pd.Series({rid: s for rid, (o, s) in res.items()})
    return pred

v1 = run("v1")
print(v1.status.str.split(":").str[0].value_counts().to_string())
results["version 1"] = score(v1)
results

In [ ]:
# YOUR CODE HERE

## 5. Probe the failures

Look at the reports where the model is wrong. Check three kinds in particular: negation ("was not admitted"), timing (a relative onset such as "about five days ago", or a past hospitalization), and reports whose text contains instructions. Then compare the rerun with the first run to measure consistency.

In [ ]:
# YOUR CODE HERE

## 6. Regression-test a new version

Score version 2's outputs with the same harness, unchanged. Compare field by field, and look at the same probes. Would you approve the upgrade?

In [ ]:
# YOUR CODE HERE

## Interpretation

Cite your own output in each answer.

**1. Did the model beat the baseline?** By field. Where did rules do as well or better?

*Your answer here.*

**2. How did it fail?** Counts of fabrication, omission, negation errors, and injected reports that succeeded.

*Your answer here.*

**3. Would you approve the new version?** Give the fields that improved and regressed, and your decision.

*Your answer here.*

**4. Where does the person sit?** For each field, name the oversight design and who is accountable for the output.

*Your answer here.*

## AI-use note (required)

Say which AI tools you used, for what, and one thing you checked or corrected in their output. "None" is an acceptable answer.

*Your note here.*

## Submit

1. Runtime → Restart session and run all. Fix anything that fails.
2. File → Save a copy in GitHub, into `labs/lab11/` of your course repository.